# 02 · Preprocessing and evaluation

## Load data

In [4]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv")

## Remove repeated rows before splitting

Keep one copy of each repeated row so identical records cannot appear in both sets. Without session IDs, some could be separate sessions. The raw CSV stays unchanged.

In [5]:
rows_before = len(df)
df = df.drop_duplicates()
pd.Series({
    "Original rows": rows_before,
    "Repeated rows removed": rows_before - len(df),
    "Rows kept": len(df),
})

Original rows            12330
Repeated rows removed      125
Rows kept                12205
dtype: int64

## Stratified 80/20 train/test split

Stratification keeps a similar share of purchases in both sets. Use `random_state=42` to repeat the same split. Keep the test set aside; preprocessing and model selection will use training data only.

In [6]:
X = df.drop(columns="Revenue")
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(df)

pd.DataFrame({
    "sessions": [len(y_train), len(y_test)],
    "purchases": [y_train.sum(), y_test.sum()],
    "purchase_percentage": [y_train.mean() * 100, y_test.mean() * 100],
}, index=["Train", "Test"]).round(2)

,sessions,purchases,purchase_percentage
Train,9764,1526,15.63
Test,2441,382,15.65
